# MathPhysics 文字题选题审计

## tl;dr

此笔记本是分支 `feature/curated-word-problems-20261005` 的可复算伴随文件。它从三个完整分类台账重算数量、分类覆盖、重复配额和推荐清单，核对报告是否一致。

**“规则检查通过”不代表全部题目已逐题通过语义审核。** 推荐题是待中文化及内容复核的候选；所有 `publish_allowed` 与 `human_approved` 应为 `false`。下方数量来自实际执行，未手工填写。

## Context & Methods

一条来源记录的主键是 `bank:source_id`。ASDiv 与 SVAMP 合为基础选题池，GSM8K 单列。`keep / adapt / hold / reject` 是全量台账的互斥处置状态；`recommended` 是其中经过配额筛选的子集，不能再次加到总题数中。

### Key Assumptions

分类体系、巩固/提高层级、每覆盖单元最多三个不同故事家族及每家族最多五题，均为项目规则。故事家族来自文本相似证据，既不是已核验题源血缘，也不能穷尽所有语义重复。SVAMP 的 V 标签只说明教学方向候选；没有配对证据时，不把 V04、V06 补成已验证覆盖。GSM8K 的原 test 不是本项目已经独立划分的考试集。

默认只读仓库内 `curated` 及锁定回执，不下载题文。若已在本机取得锁定源，可将下格 `SOURCE_ROOT` 设为其路径，额外复算所有源文件字节指纹和逐题文本指纹。

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import html
import sys
from IPython.display import display, HTML, Markdown

# 从仓库根或任意子目录打开都能定位；不依赖机器的绝对路径。
REPO = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "question-banks/SOURCE_LOCK.json").is_file())
CURATED_DIR = REPO / "question-banks/curated"
SOURCE_ROOT = None  # 可选，例如 REPO / ".selection-sources"
sys.path.insert(0, str(REPO / "question-banks/selection"))
from validate_audit import input_snapshot, reconcile, source_checks, item_key, pool

snapshot = input_snapshot(CURATED_DIR)
rows = snapshot["rows"]
selected = [row for row in rows if row["recommended"]]
checks = reconcile(snapshot)
if SOURCE_ROOT is not None:
    checks += source_checks(snapshot, SOURCE_ROOT)

def table(columns, values, caption=None):
    head = "".join(f"<th>{html.escape(str(value))}</th>" for value in columns)
    body = "".join("<tr>" + "".join(f"<td>{html.escape(str(value))}</td>" for value in row) + "</tr>" for row in values)
    title = f"<caption>{html.escape(caption)}</caption>" if caption else ""
    display(HTML(f"<div class='audit-table'><table>{title}<thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>"))

display(HTML("""<style>
.audit-table table {border-collapse:collapse; font-size:14px; line-height:1.5; margin:8px 0 20px;}
.audit-table th,.audit-table td {padding:7px 11px; border-bottom:1px solid #dfe4e9; text-align:left; vertical-align:top; white-space:normal; overflow-wrap:anywhere; max-width:420px;}
.audit-table th {background:#eef3f7; color:#17324d;}
.audit-table tr:nth-child(even) {background:#fafbfd;}
.audit-table caption {text-align:left; font-weight:600; margin-bottom:8px;}
</style>"""))


## Data

### 输入与来源

下表显示本次真正读取的台账和报告的 SHA-256。源数据的固定 commit 与文件指纹见 `question-banks/SOURCE_LOCK.json`；来源回执与该锁定文件的一致性由审计脚本核对。默认运行没有重新下载或再次读取源题文。

In [2]:
input_names = ["asdiv_selection.jsonl", "svamp_selection.jsonl", "gsm8k_selection.jsonl", "selection_report.json"]
table(["输入文件", "SHA-256"], [(name, hashlib.sha256((CURATED_DIR / name).read_bytes()).hexdigest()) for name in input_names])
display(Markdown(f"基线提交：`{snapshot['report']['baseline_commit']}`；审计日期：{snapshot['report']['audit_date']}。"))
if SOURCE_ROOT is None:
    display(Markdown("原始来源检查：本次仅核对锁定记录及既有回执；未在本笔记本内重新验证源字节。"))
else:
    display(Markdown("原始来源检查：已额外复算源文件 SHA-256、来源 ID 完整性和逐题题文指纹。"))


输入文件,SHA-256
asdiv_selection.jsonl,909b3c830075827888fa751ff5ffee6b0e84e08676ad05c79afca5a8838bb60b
svamp_selection.jsonl,38e2be253e64f38783ad7e81ae8b974b6a0258d98f483d15a1b086bb6756f982
gsm8k_selection.jsonl,9bca8776a61dc5f1a268ffb956947fb5395e6797d6e40939091c352367b5cd82
selection_report.json,a1cfc43d732dca40623a5120b8909a946b60102ed4c757a02aab90464fe50c8f


基线提交：`84fa590952c829b959177b0d93d9ce9b1031436a`；审计日期：2026-10-05。

原始来源检查：本次仅核对锁定记录及既有回执；未在本笔记本内重新验证源字节。

### 全量台账与推荐子集

“合格待中文化”和“需要改编”仍须进行中文题面、答案类型及逐题讲解审核。备用题保留在完整台账；重复控制不会把所有同知识点题都删除。

In [3]:
summary_rows = []
for bank in ("asdiv", "svamp", "gsm8k"):
    subset = [row for row in rows if row["bank"] == bank]
    decisions = Counter(row["decision"] for row in subset)
    summary_rows.append([bank.upper(), len(subset), decisions["keep"], decisions["adapt"], decisions["hold"], decisions["reject"], sum(row["recommended"] for row in subset), sum(row["publish_allowed"] for row in subset)])
table(["题库", "来源总量", "合格待中文化", "需要改编", "暂缓", "拒收/精确重复", "代表候选", "允许发布"], summary_rows)
passed = sum(check["passed"] for check in checks)
display(Markdown(f"本次读取 **{len(rows):,}** 条来源记录，代表候选 **{len(selected):,}** 条；程序一致性检查 **{passed}/{len(checks)}** 项通过。个别内容复核与自动疑点的计数均不能解释为全库人工审核率或原库错题率。"))


题库,来源总量,合格待中文化,需要改编,暂缓,拒收/精确重复,代表候选,允许发布
ASDIV,2305,1869,249,186,1,424,0
SVAMP,1000,776,77,28,119,193,0
GSM8K,8792,7180,1231,381,0,781,0


本次读取 **12,097** 条来源记录，代表候选 **1,398** 条；程序一致性检查 **29/29** 项通过。个别内容复核与自动疑点的计数均不能解释为全库人工审核率或原库错题率。

## Results

### 1. 完整性、分类边界与处置状态

下表按目的归并检查结果。完整检查清单和可选源文件验证可在命令行运行 `python question-banks/selection/validate_audit.py --source-root .selection-sources`。

In [4]:
check_groups = {
    "来源完整性、政策与溯源": [0, 1, 2, 3, 24, 25, 26],
    "题文保留边界与分类字段": [4, 5, 6],
    "发布许可与推荐资格": [7, 8, 9, 10],
    "代表清单逐题一致性": [11, 12, 13, 14],
    "GSM8K 来源 test 保留": [15, 16],
    "故事与覆盖配额": [17, 18, 19, 20, 21],
    "分库与分类统计复算": [22, 23],
    "家族证据成员对齐": [27, 28],
}
check_rows = []
for label, indexes in check_groups.items():
    group = [checks[index] for index in indexes]
    check_rows.append([label, len(group), "通过" if all(check["passed"] for check in group) else "存在失败"])
if len(checks) > 29:
    group = checks[29:]
    check_rows.append(["可选原始来源复核", len(group), "通过" if all(check["passed"] for check in group) else "存在失败"])
table(["复核内容", "检查项", "结果"], check_rows)
failures = [check["check"] for check in checks if not check["passed"]]
assert not failures, "一致性检查失败：" + "；".join(failures)


复核内容,检查项,结果
来源完整性、政策与溯源,7,通过
题文保留边界与分类字段,3,通过
发布许可与推荐资格,4,通过
代表清单逐题一致性,4,通过
GSM8K 来源 test 保留,2,通过
故事与覆盖配额,5,通过
分库与分类统计复算,2,通过
家族证据成员对齐,2,通过


### 2. 教学分类覆盖

逐条台账重新汇总，不直接抄报告数字。每题只有一个主分类，知识与推理标签可有多个。表中零值表示本次没有相应候选；不代表该方向不存在，也不以自动生成题补齐。

In [5]:
category_rows = []
for bank in ("asdiv", "svamp", "gsm8k"):
    for category, label in snapshot["taxonomy"][bank].items():
        subset = [row for row in rows if row["bank"] == bank and row["category"] == category]
        category_rows.append([bank.upper(), category, label, len(subset), sum(row["recommended"] for row in subset)])
table(["题库", "代码", "教学主分类", "全量", "代表候选"], category_rows)


题库,代码,教学主分类,全量,代表候选
ASDIV,A01,总量与部分量,446,34
ASDIV,A02,初始量、变化量与最终量,337,37
ASDIV,A03,加法比较关系,129,24
ASDIV,A04,等量分组,325,18
ASDIV,A05,倍数关系,31,22
ASDIV,A06,两步混合数量关系,94,38
ASDIV,A07,单价、数量、总价与找零,174,80
ASDIV,A08,常用单位换算应用,25,18
ASDIV,A09,时刻、经过时间与周期内安排,17,13
ASDIV,A10,分数意义、部分与整体,23,18


### 3. 重复控制与推荐上限

以下是实际推荐清单的最大值。上限约束用于练习组织，不能据此声称语义重复已全部排除。基础池与 GSM8K 分别计配额；同故事的不同语义目标可以保留，目标由 `semantic_variant_id` 与数值域区分。覆盖单元限制的是不同故事家族数，不是题数。跨池故事关联另外保留在 `assessment_group_id`。

In [6]:
cell_families = defaultdict(set)
for row in selected:
    cell_families[row["coverage_cell_id"]].add(row["story_family_id"])
family_counts = Counter((pool(row), row["story_family_id"]) for row in selected)
target_counts = Counter((pool(row), row["story_family_id"], row.get("semantic_variant_id", row["coverage_cell_id"]), row["numeric_domain"]) for row in selected)
duplicate_counts = Counter((pool(row), row["duplicate_group"]) for row in selected if row.get("duplicate_group"))
policy = snapshot["policy"]
table(["约束", "实际最大值", "允许上限"], [
    ["每覆盖单元的不同故事家族", max(map(len, cell_families.values()), default=0), policy["representative_limit_per_cell"]],
    ["每故事家族的代表题", max(family_counts.values(), default=0), policy["maximum_questions_per_story_family"]],
    ["同故事、同语义目标及数值域", max(target_counts.values(), default=0), policy["maximum_same_story_and_learning_target"]],
    ["同一重复模板组", max(duplicate_counts.values(), default=0), 1],
])


约束,实际最大值,允许上限
每覆盖单元的不同故事家族,3,3
每故事家族的代表题,5,5
同故事、同语义目标及数值域,1,1
同一重复模板组,1,1


### 4. GSM8K 来源 test 与后续考核隔离

原 test 全量保留分类，代表清单使用 train。跨 train/test 的文本家族仍是后续按 `assessment_group_id` 划分训练与考核时需要处理的关联；“本轮不推荐 test”不会自动消除这种关联。

In [7]:
gsm_rows = [row for row in rows if row["bank"] == "gsm8k"]
table(["来源划分", "台账保留", "代表候选", "暂缓或拒收"], [
    [split, sum(row["source_split"] == split for row in gsm_rows), sum(row["source_split"] == split and row["recommended"] for row in gsm_rows), sum(row["source_split"] == split and row["decision"] in {"hold", "reject"} for row in gsm_rows)]
    for split in ("train", "test")
])
family_splits = defaultdict(set)
for row in gsm_rows:
    family_splits[row["assessment_group_id"]].add(row["source_split"])
overlap = sum(splits == {"train", "test"} for splits in family_splits.values())
display(Markdown(f"逐题台账中，有 **{overlap}** 个推定文本家族同时关联来源 train 与 test。此计数是文本关联线索，不是题源亲缘的确认。"))


来源划分,台账保留,代表候选,暂缓或拒收
train,7473,781,324
test,1319,0,57


逐题台账中，有 **3** 个推定文本家族同时关联来源 train 与 test。此计数是文本关联线索，不是题源亲缘的确认。

### 5. 带来源 ID 的核对案例

此处只显示版本化分类与处理结果，以及简短核对主题，不复印原题。案例代表已发现的不同问题边界；其余记录不能因案例通过而被推定全部正确。

In [8]:
case_topics = {
    "asdiv:nluds-0016": "初始量未知，不能仅按加法归入直接巩固",
    "asdiv:nluds-0030": "选择对象答案与货币单位改编",
    "asdiv:nluds-0778": "全部人员容纳的进一取整",
    "asdiv:nluds-0881": "总量与剩余反求，不是数位专题",
    "svamp:chal-50": "条件与所问对象不足",
    "svamp:chal-588": "负中间算式不等于故事库存为负",
    "svamp:chal-680": "源公式与答案不一致",
    "gsm8k:train-0011": "和倍反求与儿童情境改编",
    "gsm8k:train-0030": "无注记的百分比基数反求",
    "gsm8k:test-0089": "来源 test 保留与解析数字笔误",
}
by_key = {item_key(row): row for row in rows}
case_rows = []
for key, topic in case_topics.items():
    row = by_key[key]
    case_rows.append([key, f"{row['category']} / {row['level']}", row["decision"], "是" if row["recommended"] else "否", ", ".join(row["quality_flags"]) or "—", topic])
table(["来源 ID", "分类/层级", "处置", "推荐", "问题标记", "核对主题"], case_rows)


来源 ID,分类/层级,处置,推荐,问题标记,核对主题
asdiv:nluds-0016,A02 / I1,keep,否,—,初始量未知，不能仅按加法归入直接巩固
asdiv:nluds-0030,A16 / I1,adapt,是,Q07,选择对象答案与货币单位改编
asdiv:nluds-0778,A13 / I1,hold,否,Q03,全部人员容纳的进一取整
asdiv:nluds-0881,A01 / C2,keep,是,—,总量与剩余反求，不是数位专题
svamp:chal-50,V02 / I1,reject,否,"Q01, Q03",条件与所问对象不足
svamp:chal-588,V00 / C2,keep,否,—,负中间算式不等于故事库存为负
svamp:chal-680,V00 / C2,reject,否,Q03,源公式与答案不一致
gsm8k:train-0011,G03 / I1,adapt,是,Q07,和倍反求与儿童情境改编
gsm8k:train-0030,G09 / I2,keep,是,—,无注记的百分比基数反求
gsm8k:test-0089,G03 / I1,adapt,否,Q03,来源 test 保留与解析数字笔误


## Takeaways

代表清单与完整台账承担不同用途：前者控制本轮练习密度，后者保留来源与备用题。分类、配额和汇总的可复算性已由上方程序检查；具体题意、所有变式关系、中文改编质量及课程年级映射仍有明确边界。

判断某题是否应进入儿童练习，应一起读取 `decision`、`quality_flags`、严重性/证据、主分类与推理标签，不能只看 `recommended`。这些台账不赋予发布许可。未来建立独立考核集时，应按故事关联家族划分，而不能直接把原 GSM8K test 当作已排除近重复的项目考核集。